# Homework GraphFrames Part 2 Consumer

In [1]:
import json
import sys
import time
from collections import defaultdict

sys.path.insert(0, "/home/hadoop/homework/spark-graphframe")

from kafka import KafkaConsumer, TopicPartition
from pyspark.sql import functions as F

from spark_graphframe_homework import (
    BOOTSTRAP_SERVERS,
    TOPIC_NAME,
    build_realtime_route_graph,
    create_kafka_topic,
    create_spark_session,
    distance_between_stations_km,
    load_station_lookup,
    read_latest_run_id,
    route_count_rows,
)


In [2]:
create_kafka_topic(TOPIC_NAME)

RUN_ID = read_latest_run_id()
MAX_MESSAGES = 250
spark = create_spark_session("graphframes-homework-consumer")
station_lookup = load_station_lookup()

{"run_id": RUN_ID, "max_messages": MAX_MESSAGES, "known_stations": len(station_lookup)}


26/03/17 16:34:38 WARN Utils: Your hostname, bigdata resolves to a loopback address: 127.0.1.1; using 10.3.134.62 instead (on interface ens3)
26/03/17 16:34:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/03/17 16:34:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/17 16:34:41 WARN DependencyUtils: Local jar /home/hadoop/graphframes-0.8.2-spark3.1-s_2.12.jar does not exist, skipping.
26/03/17 16:34:41 INFO SparkContext: Running Spark version 3.3.1
26/03/17 16:34:41 INFO ResourceUtils: ==============================================================
26/03/17 16:34:41 INFO ResourceUtils: No custom resources configured for spark.driver.
26/03/17 16:34:41 INFO ResourceUtils: ==============================================================
26/03/17 16:34:41 INFO SparkContext: Submitted application: graphframes-homework-consumer
26/03/17 16:34:42 INFO ResourceProfile: Default Resource

{'run_id': 'trip-run-20260317T161327Z',
 'max_messages': 250,
 'known_stations': 70}

In [3]:
def consume_trip_stream(run_id=RUN_ID, max_messages=MAX_MESSAGES, consumer_timeout_ms=5000):
    consumer = KafkaConsumer(
        bootstrap_servers=[BOOTSTRAP_SERVERS],
        enable_auto_commit=False,
        consumer_timeout_ms=consumer_timeout_ms,
        value_deserializer=lambda raw: json.loads(raw.decode("utf-8")),
    )

    topic_partitions = [TopicPartition(TOPIC_NAME, 0)]
    consumer.assign(topic_partitions)
    consumer.seek_to_beginning(*topic_partitions)

    route_counts = defaultdict(int)
    bike_distance_km = defaultdict(float)
    alerted_bikes = set()
    alerts = []
    processed = 0

    try:
        for record in consumer:
            message = record.value
            if run_id and message.get("run_id") != run_id:
                continue

            src = int(message["src"])
            dst = int(message["dst"])
            bike_id = int(message["bike_id"])

            if src not in station_lookup or dst not in station_lookup:
                continue

            distance_km = distance_between_stations_km(station_lookup, src, dst)
            route_counts[(src, dst)] += 1
            bike_distance_km[bike_id] += distance_km

            if bike_distance_km[bike_id] > 15 and bike_id not in alerted_bikes:
                alerted_bikes.add(bike_id)
                alerts.append(
                    {
                        "bike_id": bike_id,
                        "total_distance_km": round(bike_distance_km[bike_id], 3),
                        "src_name": station_lookup[src]["name"],
                        "dst_name": station_lookup[dst]["name"],
                    }
                )

            processed += 1
            if processed >= max_messages:
                break
    finally:
        consumer.close()

    route_graph = build_realtime_route_graph(spark, dict(route_counts), station_lookup)
    route_rows = route_count_rows(dict(route_counts), station_lookup)

    route_schema = "src long, src_name string, dst long, dst_name string, trip_count long, distance_km double"
    alert_schema = "bike_id long, total_distance_km double, src_name string, dst_name string"

    route_counts_df = (
        spark.createDataFrame(route_rows)
        if route_rows
        else spark.createDataFrame([], schema=route_schema)
    )
    alerts_df = (
        spark.createDataFrame(alerts)
        if alerts
        else spark.createDataFrame([], schema=alert_schema)
    )

    return {
        "processed_messages": processed,
        "route_graph": route_graph,
        "route_counts_df": route_counts_df,
        "alerts_df": alerts_df,
    }


In [4]:
consumer_result = consume_trip_stream()
consumer_result["processed_messages"]


26/03/17 16:34:49 INFO SharedState: Setting hive.metastore.warehouse.dir ('null') to the value of spark.sql.warehouse.dir.
26/03/17 16:34:49 INFO SharedState: Warehouse path is 'file:/home/hadoop/homework/spark-graphframe/spark-warehouse'.
26/03/17 16:34:54 INFO InMemoryFileIndex: It took 189 ms to list leaf files for 1 paths.
26/03/17 16:34:54 INFO InMemoryFileIndex: It took 2 ms to list leaf files for 1 paths.
26/03/17 16:35:02 INFO FileSourceStrategy: Pushed Filters: 
26/03/17 16:35:02 INFO FileSourceStrategy: Post-Scan Filters: (length(trim(value#0, None)) > 0)
26/03/17 16:35:02 INFO FileSourceStrategy: Output Data Schema: struct<value: string>
26/03/17 16:35:06 INFO CodeGenerator: Code generated in 1593.188268 ms
26/03/17 16:35:07 INFO MemoryStore: Block broadcast_0 stored as values in memory (estimated size 351.2 KiB, free 366.0 MiB)
26/03/17 16:35:08 INFO MemoryStore: Block broadcast_0_piece0 stored as bytes in memory (estimated size 34.2 KiB, free 365.9 MiB)
26/03/17 16:35:08 I

[Stage 0:>                                                          (0 + 1) / 1]

26/03/17 16:35:09 INFO TaskSetManager: Starting task 0.0 in stage 0.0 (TID 0) (server1, executor driver, partition 0, PROCESS_LOCAL, 4933 bytes) taskResourceAssignments Map()
26/03/17 16:35:09 INFO Executor: Running task 0.0 in stage 0.0 (TID 0)
26/03/17 16:35:12 INFO FileScanRDD: Reading File path: file:///home/hadoop/homework/spark-graphframe/data/station.csv, range: 0-5272, partition values: [empty row]
26/03/17 16:35:12 INFO CodeGenerator: Code generated in 59.558821 ms
26/03/17 16:35:13 INFO Executor: Finished task 0.0 in stage 0.0 (TID 0). 1617 bytes result sent to driver
26/03/17 16:35:13 INFO TaskSetManager: Finished task 0.0 in stage 0.0 (TID 0) in 3366 ms on server1 (executor driver) (1/1)
26/03/17 16:35:13 INFO TaskSchedulerImpl: Removed TaskSet 0.0, whose tasks have all completed, from pool 
26/03/17 16:35:13 INFO DAGScheduler: ResultStage 0 (csv at NativeMethodAccessorImpl.java:0) finished in 3.974 s
26/03/17 16:35:13 INFO DAGScheduler: Job 0 is finished. Cancelling potent

26/03/17 16:35:13 INFO FileSourceStrategy: Pushed Filters: 
26/03/17 16:35:13 INFO FileSourceStrategy: Post-Scan Filters: 
26/03/17 16:35:13 INFO FileSourceStrategy: Output Data Schema: struct<value: string>
26/03/17 16:35:13 INFO MemoryStore: Block broadcast_2 stored as values in memory (estimated size 351.2 KiB, free 365.6 MiB)
26/03/17 16:35:13 INFO MemoryStore: Block broadcast_2_piece0 stored as bytes in memory (estimated size 34.2 KiB, free 365.5 MiB)
26/03/17 16:35:13 INFO BlockManagerInfo: Added broadcast_2_piece0 in memory on server1:41763 (size: 34.2 KiB, free: 366.2 MiB)
26/03/17 16:35:13 INFO SparkContext: Created broadcast 2 from csv at NativeMethodAccessorImpl.java:0
26/03/17 16:35:13 INFO FileSourceScanExec: Planning scan with bin packing, max size: 4194304 bytes, open cost is considered as scanning 4194304 bytes.


/home/hadoop/homework/.venv/lib/python3.10/site-packages/pyspark/sql/dataframe.py:148: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(


250

In [5]:
if consumer_result["processed_messages"] == 0:
    print("No matching Kafka messages were consumed. Run producer.ipynb first.")
else:
    trip_summary_df = (
        consumer_result["route_graph"]
        .triplets
        .select(
            F.col("edge.trip_count").alias("trip_count"),
            F.col("src.name").alias("from_station"),
            F.col("dst.name").alias("to_station"),
            F.col("edge.distance_km").alias("distance_km"),
        )
        .orderBy(F.desc("trip_count"), F.desc("distance_km"))
    )
    trip_summary_df.show(10, truncate=False)


/home/hadoop/homework/.venv/lib/python3.10/site-packages/pyspark/sql/dataframe.py:127: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


26/03/17 16:35:19 INFO BlockManagerInfo: Removed broadcast_1_piece0 on server1:41763 in memory (size: 5.9 KiB, free: 366.2 MiB)
26/03/17 16:35:19 INFO FileSourceStrategy: Pushed Filters: 
26/03/17 16:35:19 INFO FileSourceStrategy: Post-Scan Filters: atleastnnonnulls(3, cast(station_id#17 as bigint), cast(lat#19 as double), cast(long#20 as double)),isnotnull(cast(station_id#17 as bigint))
26/03/17 16:35:19 INFO FileSourceStrategy: Output Data Schema: struct<station_id: string, name: string, lat: string, long: string ... 2 more fields>
26/03/17 16:35:19 INFO FileSourceStrategy: Pushed Filters: 
26/03/17 16:35:19 INFO FileSourceStrategy: Post-Scan Filters: atleastnnonnulls(3, cast(station_id#95 as bigint), cast(lat#97 as double), cast(long#98 as double)),isnotnull(cast(station_id#95 as bigint))
26/03/17 16:35:19 INFO FileSourceStrategy: Output Data Schema: struct<station_id: string, name: string, lat: string, long: string ... 2 more fields>
26/03/17 16:35:20 INFO CodeGenerator: Code gener

[Stage 1:>                                                          (0 + 1) / 1]

26/03/17 16:35:21 INFO CodeGenerator: Code generated in 19.305347 ms
26/03/17 16:35:21 INFO CodeGenerator: Code generated in 23.765549 ms
26/03/17 16:35:21 INFO CodeGenerator: Code generated in 16.682844 ms
26/03/17 16:35:21 INFO CodeGenerator: Code generated in 16.514354 ms
26/03/17 16:35:21 INFO CodeGenerator: Code generated in 30.688369 ms
26/03/17 16:35:21 INFO Executor: Finished task 0.0 in stage 1.0 (TID 1). 2424 bytes result sent to driver
26/03/17 16:35:21 INFO TaskSetManager: Finished task 0.0 in stage 1.0 (TID 1) in 804 ms on server1 (executor driver) (1/1)
26/03/17 16:35:21 INFO TaskSchedulerImpl: Removed TaskSet 1.0, whose tasks have all completed, from pool 
26/03/17 16:35:21 INFO DAGScheduler: ShuffleMapStage 1 (showString at NativeMethodAccessorImpl.java:0) finished in 0.898 s
26/03/17 16:35:21 INFO DAGScheduler: looking for newly runnable stages
26/03/17 16:35:21 INFO DAGScheduler: running: Set()
26/03/17 16:35:21 INFO DAGScheduler: waiting: Set()
26/03/17 16:35:21 INFO

26/03/17 16:35:22 INFO CodeGenerator: Code generated in 18.166872 ms
26/03/17 16:35:22 INFO MemoryStore: Block broadcast_8 stored as values in memory (estimated size 1024.7 KiB, free 363.0 MiB)
26/03/17 16:35:22 INFO MemoryStore: Block broadcast_7 stored as values in memory (estimated size 1024.7 KiB, free 364.0 MiB)
26/03/17 16:35:22 INFO MemoryStore: Block broadcast_7_piece0 stored as bytes in memory (estimated size 2.5 KiB, free 363.0 MiB)
26/03/17 16:35:22 INFO MemoryStore: Block broadcast_8_piece0 stored as bytes in memory (estimated size 2.5 KiB, free 363.0 MiB)
26/03/17 16:35:22 INFO BlockManagerInfo: Added broadcast_7_piece0 in memory on server1:41763 (size: 2.5 KiB, free: 366.1 MiB)
26/03/17 16:35:22 INFO BlockManagerInfo: Added broadcast_8_piece0 in memory on server1:41763 (size: 2.5 KiB, free: 366.1 MiB)
26/03/17 16:35:22 INFO SparkContext: Created broadcast 7 from $anonfun$withThreadLocalCaptured$1 at FutureTask.java:266
26/03/17 16:35:22 INFO SparkContext: Created broadcas

[Stage 5:>                                                          (0 + 4) / 8]

26/03/17 16:35:26 INFO PythonRunner: Times: total = 2388, boot = 2101, init = 287, finish = 0
26/03/17 16:35:26 INFO PythonRunner: Times: total = 2466, boot = 2156, init = 310, finish = 0
26/03/17 16:35:26 INFO Executor: Finished task 3.0 in stage 5.0 (TID 7). 5843 bytes result sent to driver
26/03/17 16:35:26 INFO Executor: Finished task 0.0 in stage 5.0 (TID 4). 6061 bytes result sent to driver
26/03/17 16:35:26 INFO PythonRunner: Times: total = 2515, boot = 2123, init = 392, finish = 0
26/03/17 16:35:26 INFO TaskSetManager: Starting task 4.0 in stage 5.0 (TID 8) (server1, executor driver, partition 4, PROCESS_LOCAL, 5080 bytes) taskResourceAssignments Map()
26/03/17 16:35:26 INFO Executor: Running task 4.0 in stage 5.0 (TID 8)
26/03/17 16:35:26 INFO Executor: Finished task 1.0 in stage 5.0 (TID 5). 5832 bytes result sent to driver
26/03/17 16:35:26 INFO TaskSetManager: Starting task 5.0 in stage 5.0 (TID 9) (server1, executor driver, partition 5, PROCESS_LOCAL, 5080 bytes) taskResou

[Stage 5:=============================>                             (4 + 4) / 8]

26/03/17 16:35:26 INFO TaskSetManager: Starting task 7.0 in stage 5.0 (TID 11) (server1, executor driver, partition 7, PROCESS_LOCAL, 5187 bytes) taskResourceAssignments Map()
26/03/17 16:35:26 INFO Executor: Running task 7.0 in stage 5.0 (TID 11)
26/03/17 16:35:26 INFO TaskSetManager: Finished task 2.0 in stage 5.0 (TID 6) in 3026 ms on server1 (executor driver) (4/8)
26/03/17 16:35:26 INFO PythonRunner: Times: total = 342, boot = -47, init = 389, finish = 0
26/03/17 16:35:26 INFO Executor: Finished task 4.0 in stage 5.0 (TID 8). 5986 bytes result sent to driver
26/03/17 16:35:26 INFO TaskSetManager: Finished task 4.0 in stage 5.0 (TID 8) in 474 ms on server1 (executor driver) (5/8)
26/03/17 16:35:26 INFO PythonRunner: Times: total = 380, boot = 97, init = 282, finish = 1
26/03/17 16:35:26 INFO PythonRunner: Times: total = 484, boot = 32, init = 452, finish = 0
26/03/17 16:35:26 INFO PythonRunner: Times: total = 500, boot = 56, init = 444, finish = 0
26/03/17 16:35:26 INFO Executor: F

26/03/17 16:35:27 INFO CodeGenerator: Code generated in 114.611293 ms
26/03/17 16:35:27 INFO CodeGenerator: Code generated in 25.819507 ms
+----------+---------------------------------------------+----------------------------------------+-----------+
|trip_count|from_station                                 |to_station                              |distance_km|
+----------+---------------------------------------------+----------------------------------------+-----------+
|6         |San Francisco Caltrain 2 (330 Townsend)      |Townsend at 7th                         |0.886      |
|5         |5th at Howard                                |San Francisco Caltrain 2 (330 Townsend) |1.024      |
|4         |Steuart at Market                            |San Francisco Caltrain (Townsend at 4th)|1.95       |
|4         |Embarcadero at Sansome                       |Steuart at Market                       |1.413      |
|4         |Mountain View Caltrain Station               |Castro Street and E

In [6]:
consumer_result["alerts_df"].orderBy(F.desc("total_distance_km"), F.asc("bike_id")).show(20, truncate=False)


26/03/17 16:35:27 INFO CodeGenerator: Code generated in 35.81526 ms
26/03/17 16:35:27 INFO CodeGenerator: Code generated in 53.427991 ms
26/03/17 16:35:27 INFO SparkContext: Starting job: showString at NativeMethodAccessorImpl.java:0
26/03/17 16:35:27 INFO DAGScheduler: Got job 5 (showString at NativeMethodAccessorImpl.java:0) with 8 output partitions
26/03/17 16:35:27 INFO DAGScheduler: Final stage: ResultStage 6 (showString at NativeMethodAccessorImpl.java:0)
26/03/17 16:35:27 INFO DAGScheduler: Parents of final stage: List()
26/03/17 16:35:27 INFO DAGScheduler: Missing parents: List()
26/03/17 16:35:27 INFO DAGScheduler: Submitting ResultStage 6 (MapPartitionsRDD[43] at showString at NativeMethodAccessorImpl.java:0), which has no missing parents
26/03/17 16:35:27 INFO MemoryStore: Block broadcast_10 stored as values in memory (estimated size 15.4 KiB, free 363.0 MiB)
26/03/17 16:35:27 INFO MemoryStore: Block broadcast_10_piece0 stored as bytes in memory (estimated size 7.6 KiB, free

[Stage 6:======================>                                    (3 + 4) / 8]

26/03/17 16:35:28 INFO PythonRunner: Times: total = 756, boot = -919, init = 1675, finish = 0
26/03/17 16:35:28 INFO Executor: Finished task 3.0 in stage 6.0 (TID 15). 4507 bytes result sent to driver
26/03/17 16:35:28 INFO TaskSetManager: Starting task 7.0 in stage 6.0 (TID 19) (server1, executor driver, partition 7, PROCESS_LOCAL, 4433 bytes) taskResourceAssignments Map()
26/03/17 16:35:28 INFO Executor: Running task 7.0 in stage 6.0 (TID 19)
26/03/17 16:35:28 INFO TaskSetManager: Finished task 3.0 in stage 6.0 (TID 15) in 919 ms on server1 (executor driver) (4/8)


[Stage 6:=============================>                             (4 + 4) / 8]

26/03/17 16:35:28 INFO PythonRunner: Times: total = 555, boot = -153, init = 708, finish = 0
26/03/17 16:35:28 INFO Executor: Finished task 5.0 in stage 6.0 (TID 17). 4464 bytes result sent to driver
26/03/17 16:35:28 INFO TaskSetManager: Finished task 5.0 in stage 6.0 (TID 17) in 661 ms on server1 (executor driver) (5/8)
26/03/17 16:35:28 INFO PythonRunner: Times: total = 619, boot = 30, init = 589, finish = 0
26/03/17 16:35:28 INFO Executor: Finished task 4.0 in stage 6.0 (TID 16). 4464 bytes result sent to driver
26/03/17 16:35:28 INFO TaskSetManager: Finished task 4.0 in stage 6.0 (TID 16) in 714 ms on server1 (executor driver) (6/8)
26/03/17 16:35:29 INFO PythonRunner: Times: total = 665, boot = 66, init = 599, finish = 0
26/03/17 16:35:29 INFO Executor: Finished task 6.0 in stage 6.0 (TID 18). 4464 bytes result sent to driver
26/03/17 16:35:29 INFO TaskSetManager: Finished task 6.0 in stage 6.0 (TID 18) in 794 ms on server1 (executor driver) (7/8)
26/03/17 16:35:29 INFO PythonRun

[Stage 6:===================================================>       (7 + 1) / 8]

26/03/17 16:35:29 INFO TaskSetManager: Finished task 7.0 in stage 6.0 (TID 19) in 684 ms on server1 (executor driver) (8/8)
26/03/17 16:35:29 INFO TaskSchedulerImpl: Removed TaskSet 6.0, whose tasks have all completed, from pool 
26/03/17 16:35:29 INFO DAGScheduler: ResultStage 6 (showString at NativeMethodAccessorImpl.java:0) finished in 1.599 s
26/03/17 16:35:29 INFO DAGScheduler: Job 5 is finished. Cancelling potential speculative or zombie tasks for this job
26/03/17 16:35:29 INFO TaskSchedulerImpl: Killing all running tasks in stage 6: Stage finished
26/03/17 16:35:29 INFO DAGScheduler: Job 5 finished: showString at NativeMethodAccessorImpl.java:0, took 1.621626 s
26/03/17 16:35:29 INFO CodeGenerator: Code generated in 86.615242 ms
+-------+-----------------+--------+--------+
|bike_id|total_distance_km|src_name|dst_name|
+-------+-----------------+--------+--------+
+-------+-----------------+--------+--------+

